In [1]:
from transformers import AutoModel, AutoTokenizer, AutoProcessor
from PIL import Image
import os
import polars as pl
import torch
from torch.utils.data import DataLoader
import sys
sys.path.append("..")
from src.datasets.mp16 import MP16Dataset
import  torch.nn.functional as F
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model_name = "openai/clip-vit-large-patch14"
base_img_path = "../../datasets/google-landmark/index-img"

prompt_template = "a photo of landmark {label}"

In [2]:
clip_processor = AutoProcessor.from_pretrained(clip_model_name)
clip_tokenizer = AutoTokenizer.from_pretrained(clip_model_name)
clip_model = AutoModel.from_pretrained(clip_model_name).to(device)

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
df = pl.read_csv("../../datasets/google-landmark/train_ref.csv")
dataset = MP16Dataset(df, img_col="id", img_base_path=base_img_path)

In [4]:
with open("./labels.txt", "r") as f:
    labels = [lb.strip("\n") for lb in f.readlines()]
    labels_prompt = [prompt_template.format(label=lb) for lb in labels]

# add fallback
labels.append("other")
labels_prompt.append("a photo of an indoor scene, a mundane household object, or a generic texture that is not an exterior landmark.")

In [5]:
labels_prompt

['a photo of landmark tower',
 'a photo of landmark lighthouse',
 'a photo of landmark skyscraper',
 'a photo of landmark bridge',
 'a photo of landmark triumphal_arch',
 'a photo of landmark column_arch',
 'a photo of landmark gate',
 'a photo of landmark castle',
 'a photo of landmark fortress',
 'a photo of landmark ruins',
 'a photo of landmark statue',
 'a photo of landmark pillar_monument',
 'a photo of landmark fountain',
 'a photo of landmark park',
 'a photo of landmark mountain_peak',
 'a photo of landmark waterfall',
 'a photo of landmark rock_formation',
 'a photo of landmark aerial_view',
 'a photo of landmark street_scene',
 'a photo of an indoor scene, a mundane household object, or a generic texture that is not an exterior landmark.']

In [6]:
with torch.no_grad():
    label_tokenized = clip_tokenizer(labels_prompt, return_tensors="pt", padding=True)
    label_embedding = clip_model.get_text_features(**{k:v.to(device) for k, v in label_tokenized.items()})
    label_embedding = F.normalize(label_embedding.pooler_output.cpu())

In [7]:
def collate_fn(batch):
    images = [b["image"] for b in batch]
    img_processed = clip_processor(images=images, return_tensors="pt")
    return img_processed
loader = DataLoader(dataset, batch_size=1024, collate_fn=collate_fn)

In [8]:
label_preds = []

with torch.no_grad():
    for batch in tqdm(loader, desc="zero-shot classify"):
        img_embedding = clip_model.get_image_features(**{k:v.to(device) for k, v in batch.items()})
        img_embedding = F.normalize(img_embedding.pooler_output.cpu())
        
        similarities = img_embedding @ label_embedding.T
        predicted = torch.argmax(similarities.softmax(dim=-1), dim=-1).tolist()
        label_preds.extend(predicted)

        torch.cuda.empty_cache()
        del img_embedding
        del similarities

zero-shot classify: 100%|██████████| 1781/1781 [6:10:32<00:00, 12.48s/it]  


In [12]:
df_pred = df.with_columns(pl.Series("pred_label", [labels[lp] for lp in label_preds]))

In [20]:
df_pred.filter(pl.col("pred_label") != "other")

landmark_id,input_url,resolved_url,geohack_url,latitude,longitude,error,supercategory,hierarchical_label,natural_or_human_made,rg_name,rg_admin1,rg_admin2,country_code,country,continent,subregion,id,url,pred_label
i64,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
104169,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",56.123889,-3.947778,null,"""castle""","""castle / fort""","""human-made""","""Stirling""","""Scotland""","""Stirling""","""GB""","""United Kingdom""","""Europe""","""Northern Europe""","""202cd79556f30760""","""http://upload.wikimedia.org/wi…","""castle"""
2474,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",51.685278,-2.543611,null,"""river""","""river""","""natural""","""Hill""","""England""","""South Gloucestershire""","""GB""","""United Kingdom""","""Europe""","""Northern Europe""","""4072182eddd0100e""","""https://upload.wikimedia.org/w…","""waterfall"""
6888,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",44.468333,34.144444,null,"""park""","""parks""","""natural""","""Livadiya""","""Crimea""","""""","""UA""","""Ukraine""","""Europe""","""Eastern Europe""","""6f31b874d1a4d489""","""https://upload.wikimedia.org/w…","""park"""
25719,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",45.58359,9.27567,null,"""cathedral""","""church""","""human-made""","""Monza""","""Lombardy""","""Provincia di Monza e Brianza""","""IT""","""Italy""","""Europe""","""Southern Europe""","""16d8aa057cdd01b9""","""http://upload.wikimedia.org/wi…","""column_arch"""
122849,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",47.6558,8.68298,null,"""observation tower""","""tower""","""human-made""","""Marthalen""","""Zurich""","""Bezirk Andelfingen""","""CH""","""Switzerland""","""Europe""","""Western Europe""","""3968e37e503f3109""","""https://upload.wikimedia.org/w…","""castle"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
46054,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",-14.333333,-68.333333,null,"""national park""","""parks""","""natural""","""Rurrenabaque""","""El Beni""","""""","""BO""","""Bolivia""","""Americas""","""South America""","""b4e2a7c6c2a2c899""","""https://upload.wikimedia.org/w…","""mountain_peak"""
49555,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",51.4457,-0.0779,null,"""area of London""","""city""","""human-made""","""Camberwell""","""England""","""Greater London""","""GB""","""United Kingdom""","""Europe""","""Northern Europe""","""9b0e881f2ead47d1""","""https://upload.wikimedia.org/w…","""street_scene"""
42142,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",38.752778,-9.184722,null,"""association football stadium""","""sports venue""","""human-made""","""Pontinha""","""Lisbon""","""Odivelas""","""PT""","""Portugal""","""Europe""","""Southern Europe""","""8877b1c333e1f1ad""","""https://upload.wikimedia.org/w…","""statue"""


In [24]:
df_pred.write_csv("../../datasets/google-landmark/train_ref_predicted.csv")

In [31]:
df_index = pl.read_csv("../../datasets/google-landmark/index_ref.csv")
dataset_index = MP16Dataset(df_index, img_col="id", img_base_path=base_img_path)
loader_index = DataLoader(dataset_index, batch_size=1024, collate_fn=collate_fn)

In [32]:
index_label_preds = []

with torch.no_grad():
    for batch in tqdm(loader_index, desc="zero-shot classify"):
        img_embedding = clip_model.get_image_features(**{k:v.to(device) for k, v in batch.items()})
        img_embedding = F.normalize(img_embedding.pooler_output.cpu())
        
        similarities = img_embedding @ label_embedding.T
        predicted = torch.argmax(similarities.softmax(dim=-1), dim=-1).tolist()
        index_label_preds.extend(predicted)

        torch.cuda.empty_cache()
        del img_embedding
        del similarities

zero-shot classify: 100%|██████████| 404/404 [1:22:14<00:00, 12.21s/it]


In [34]:
df_index_pred = df_index.with_columns(pl.Series("pred_label", [labels[lp] for lp in index_label_preds]))
df_index_pred.head()

landmark_id,input_url,resolved_url,geohack_url,latitude,longitude,error,supercategory,hierarchical_label,natural_or_human_made,rg_name,rg_admin1,rg_admin2,country_code,country,continent,subregion,id,pred_label
i64,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str
78775,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",33.333333,-117.416667,null,"""military base""",null,null,"""Camp Pendleton North""","""California""","""San Diego County""","""US""","""United States""","""Americas""","""Northern America""","""6c07a95bbed8387a""","""fortress"""
14854,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",44.1436,-73.9867,null,"""mountain""","""mountain""","""natural""","""Lake Placid""","""New York""","""Essex County""","""US""","""United States""","""Americas""","""Northern America""","""fa69da76c3eb58ea""","""mountain_peak"""
38501,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",54.6627,11.7331,null,"""town""","""city""","""human-made""","""Sakskobing""","""Zealand""","""Guldborgsund Kommune""","""DK""","""Denmark""","""Europe""","""Northern Europe""","""34f7a170186c9a60""","""street_scene"""
36227,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",38.892778,-77.023056,null,"""archive building""",null,null,"""Washington, D.C.""","""Washington, D.C.""","""""","""US""","""United States""","""Americas""","""Northern America""","""2eaf1281b9715e91""","""column_arch"""
39704,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",16.284983,121.091865,null,"""church building""","""church""","""human-made""","""Malasin""","""Cagayan Valley""","""Province of Nueva Vizcaya""","""PH""","""Philippines""","""Asia""","""Southeast Asia""","""d3b1332814fc7b80""","""gate"""


In [36]:
df_index_pred.write_csv("../../datasets/google-landmark/index_ref_predicted.csv")

In [46]:
df_pred.filter(pl.col("pred_label").is_in(labels[:12])).write_csv("../../datasets/google-landmark/train_ref_predicted_filtered.csv")
df_index_pred.filter(pl.col("pred_label").is_in(labels[:12])).write_csv("../../datasets/google-landmark/index_ref_predicted_filtered.csv")